<a href="https://colab.research.google.com/github/RishikaBajaj31/RevivePixels/blob/main/RevivePixels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn python-multipart pillow torch torchvision nest-asyncio

import torch
import torch.nn as nn
import torchvision.transforms as transforms
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import StreamingResponse
from io import BytesIO
from PIL import Image
import nest_asyncio
import uvicorn
from threading import Thread

# Define the model directly here
class SimpleSRCNN(nn.Module):
    def __init__(self):
        super(SimpleSRCNN, self).__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=9, padding=4),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, kernel_size=5, padding=2)
        )

    def forward(self, x):
        return self.layers(x)

# Init FastAPI
app = FastAPI()
model = SimpleSRCNN()
model.eval()

# Transforms
transform = transforms.Compose([transforms.ToTensor()])
to_pil = transforms.ToPILImage()

@app.post("/enhance/")
async def enhance_image(file: UploadFile = File(...)):
    image = Image.open(BytesIO(await file.read())).convert("RGB")
    input_tensor = transform(image).unsqueeze(0)

    with torch.no_grad():
        output_tensor = model(input_tensor)

    output_image = to_pil(output_tensor.squeeze(0))
    buf = BytesIO()
    output_image.save(buf, format='PNG')
    buf.seek(0)
    return StreamingResponse(buf, media_type="image/png")

# Run FastAPI inside Colab
def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

nest_asyncio.apply()
Thread(target=run).start()


In [2]:
with open("model.py", "w") as f:
    f.write('''
import torch.nn as nn

class SimpleSRCNN(nn.Module):
    def __init__(self):
        super(SimpleSRCNN, self).__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=9, padding=4),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, kernel_size=5, padding=2)
        )

    def forward(self, x):
        return self.layers(x)
''')


In [3]:
# main.py
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import StreamingResponse
from io import BytesIO
from PIL import Image
import torch
import torchvision.transforms as transforms
from model import SimpleSRCNN

app = FastAPI()

# Load model
model = SimpleSRCNN()
model.eval()

# Image pre/post-processing
transform = transforms.Compose([
    transforms.ToTensor(),
])

to_pil = transforms.ToPILImage()

@app.post("/enhance/")
async def enhance_image(file: UploadFile = File(...)):
    image = Image.open(BytesIO(await file.read())).convert("RGB")
    input_tensor = transform(image).unsqueeze(0)

    with torch.no_grad():
        output_tensor = model(input_tensor)

    output_image = to_pil(output_tensor.squeeze(0))
    buf = BytesIO()
    output_image.save(buf, format='PNG')
    buf.seek(0)
    return StreamingResponse(buf, media_type="image/png")


In [4]:
!pip uninstall -y pyngrok
!pip install pyngrok --upgrade

from pyngrok import conf, ngrok
import os

# Delete old ngrok binary if it exists
ngrok_bin = os.path.expanduser("~/.ngrok2/ngrok.yml")
if os.path.exists(ngrok_bin):
    os.remove(ngrok_bin)

# Kill any existing ngrok processes
ngrok.kill()


Found existing installation: pyngrok 7.2.3
Uninstalling pyngrok-7.2.3:
  Successfully uninstalled pyngrok-7.2.3
  Using cached pyngrok-7.2.3-py3-none-any.whl.metadata (8.7 kB)
Using cached pyngrok-7.2.3-py3-none-any.whl (23 kB)


In [5]:
# Replace with your actual token
ngrok.set_auth_token("2visowfmYMoIdekpSTqvwpOMUSq_5okknMUziK1h5EZVKcQmp")

In [6]:
# This should now work
public_url = ngrok.connect(8000)
print("FastAPI is live at:", public_url)


FastAPI is live at: NgrokTunnel: "https://9e1b-34-169-71-21.ngrok-free.app" -> "http://localhost:8000"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [7]:
import nest_asyncio
import uvicorn
from threading import Thread

nest_asyncio.apply()
Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000)).start()


In [8]:
# Setup FastAPI app
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import StreamingResponse
from io import BytesIO
from PIL import Image
import nest_asyncio
import uvicorn
from threading import Thread

# Define a simple SRCNN model
class SimpleSRCNN(nn.Module):
    def __init__(self):
        super(SimpleSRCNN, self).__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=9, padding=4),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 32, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 3, kernel_size=5, padding=2)
        )

    def forward(self, x):
        return self.layers(x)

# Initialize FastAPI and model
app = FastAPI()
model = SimpleSRCNN()
model.eval()

transform = transforms.Compose([transforms.ToTensor()])
to_pil = transforms.ToPILImage()

@app.post("/enhance/")
async def enhance_image(file: UploadFile = File(...)):
    image = Image.open(BytesIO(await file.read())).convert("RGB")
    input_tensor = transform(image).unsqueeze(0)

    with torch.no_grad():
        output_tensor = model(input_tensor)

    output_image = to_pil(output_tensor.squeeze(0))
    buf = BytesIO()
    output_image.save(buf, format='PNG')
    buf.seek(0)
    return StreamingResponse(buf, media_type="image/png")

# Start FastAPI server
def run_app():
    uvicorn.run(app, host="0.0.0.0", port=8000)

nest_asyncio.apply()
Thread(target=run_app).start()


ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-4' coro=<Server.serve() done, defined at /usr/local/lib/python3.11/dist-packages/uvicorn/server.py:68> exception=SystemExit(1)>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/uvicorn/server.py", line 163, in startup
    server = await loop.create_server(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/asyncio/base_events.py", line 1536, in create_server
    raise OSError(err.errno, msg) from None
OSError: [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "<ipython-input-7-91685f6e4144>", line 6, in <lambd

In [9]:
import os
from fastapi.staticfiles import StaticFiles
from uuid import uuid4

# Make folders to store images
os.makedirs("uploads", exist_ok=True)
os.makedirs("restored", exist_ok=True)

# Mount static folder so we can serve files
app.mount("/uploads", StaticFiles(directory="uploads"), name="uploads")
app.mount("/restored", StaticFiles(directory="restored"), name="restored")

@app.post("/enhance/")
async def enhance_image(file: UploadFile = File(...)):
    # Save uploaded file
    image_id = str(uuid4())
    original_path = f"uploads/{image_id}.png"
    restored_path = f"restored/{image_id}.png"

    image = Image.open(BytesIO(await file.read())).convert("RGB")
    image.save(original_path)

    # Run through model
    input_tensor = transform(image).unsqueeze(0)
    with torch.no_grad():
        output_tensor = model(input_tensor)

    output_image = to_pil(output_tensor.squeeze(0))
    output_image.save(restored_path)

    return {
        "original_image_url": f"/uploads/{image_id}.png",
        "restored_image_url": f"/restored/{image_id}.png"
    }


In [10]:
!pip install jinja2

In [11]:
from fastapi.responses import HTMLResponse
from fastapi.templating import Jinja2Templates
from fastapi import Request

# Create templates folder
os.makedirs("templates", exist_ok=True)

# Mount template engine
templates = Jinja2Templates(directory="templates")

@app.get("/", response_class=HTMLResponse)
def home(request: Request):
    return templates.TemplateResponse("index.html", {"request": request})


ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-7' coro=<Server.serve() done, defined at /usr/local/lib/python3.11/dist-packages/uvicorn/server.py:68> exception=SystemExit(1)>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/uvicorn/server.py", line 163, in startup
    server = await loop.create_server(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/asyncio/base_events.py", line 1536, in create_server
    raise OSError(err.errno, msg) from None
OSError: [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "<ipython-input-8-9d2aa6e166d7>", line 52, in run_a

In [12]:
@app.post("/upload/", response_class=HTMLResponse)
async def upload_image(request: Request, file: UploadFile = File(...)):
    image_id = str(uuid4())
    original_path = f"uploads/{image_id}.png"
    restored_path = f"restored/{image_id}.png"

    image = Image.open(BytesIO(await file.read())).convert("RGB")
    image.save(original_path)

    input_tensor = transform(image).unsqueeze(0)
    with torch.no_grad():
        output_tensor = model(input_tensor)

    output_image = to_pil(output_tensor.squeeze(0))
    output_image.save(restored_path)

    return templates.TemplateResponse("result.html", {
        "request": request,
        "original": f"/uploads/{image_id}.png",
        "restored": f"/restored/{image_id}.png"
    })


In [13]:
# Create the templates directory
import os
os.makedirs("templates", exist_ok=True)

# index.html (upload form)
with open("templates/index.html", "w") as f:
    f.write("""
<!DOCTYPE html>
<html>
<head>
    <title>Revive Pixels - Upload Image</title>
</head>
<body>
    <h2>Upload an Image for Restoration</h2>
    <form action="/upload/" enctype="multipart/form-data" method="post">
        <input type="file" name="file" accept="image/*" required>
        <button type="submit">Enhance</button>
    </form>
</body>
</html>
""")

# result.html (show original and enhanced images)
with open("templates/result.html", "w") as f:
    f.write("""
<!DOCTYPE html>
<html>
<head>
    <title>Enhanced Result</title>
</head>
<body>
    <h2>Before and After</h2>
    <div style="display: flex; gap: 20px;">
        <div>
            <h3>Original</h3>
            <img src="{{ original }}" width="300">
        </div>
        <div>
            <h3>Restored</h3>
            <img src="{{ restored }}" width="300">
        </div>
    </div>
    <br>
    <a href="/">🔙 Enhance another</a>
</body>
</html>
""")
